# ForkWise — Provision Infrastructure

**Fully reproducible.** Run all cells top to bottom for a fresh deployment.

**What this does:**
1. Creates a KVM@TACC lease via `chi` SDK
2. Installs Terraform, provisions 3 VMs + network + floating IP
3. Prints all commands to run on the nodes

**Prerequisites:**
- `clouds.yaml` in `infra/tf/kvm/` (KVM@TACC application credentials)
- SSH keypair `forkwise-key` registered on Chameleon

## Step 1: Configure Chameleon Context

In [1]:
from chi import server, context, lease, network
import chi, os, time, datetime, subprocess, shutil, json

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

# ─── Project configuration ───
PROJECT_PREFIX = "proj01"
SSH_KEY_NAME   = "forkwise-key"
LEASE_DAYS     = 7
LEASE_END      = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=LEASE_DAYS)
VM_FLAVOR      = "m1.medium"
VM_COUNT       = 3
VM_IMAGE       = "CC-Ubuntu24.04"

# Object store credentials (fill these in)
OS_ACCESS_KEY  = "8921c48faf83433db2b1439a9b2889fd"
OS_SECRET_KEY  = "7d1ce78efc5a48019888c9f3fa8ba2dd"

print(f"Project:    {PROJECT_PREFIX}")
print(f"Keypair:    {SSH_KEY_NAME}")
print(f"Flavor:     {VM_FLAVOR} x {VM_COUNT}")
print(f"Lease ends: {LEASE_END}")

Project:    proj01
Keypair:    forkwise-key
Flavor:     m1.medium x 3
Lease ends: 2026-05-03 05:59:10.712521+00:00


## Step 2: Install Terraform

In [2]:
TF_VERSION = "1.9.8"

commands = [
    "mkdir -p /work/.local/bin",
    f"wget -q https://releases.hashicorp.com/terraform/{TF_VERSION}/terraform_{TF_VERSION}_linux_amd64.zip",
    f"unzip -o -q terraform_{TF_VERSION}_linux_amd64.zip",
    "mv terraform /work/.local/bin",
    f"rm terraform_{TF_VERSION}_linux_amd64.zip",
]

for cmd in commands:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error: {cmd}\n{result.stderr}")

result = subprocess.run("/work/.local/bin/terraform --version", shell=True, capture_output=True, text=True)
print(result.stdout.strip().split("\n")[0])

Terraform v1.9.8


## Step 3: Create Lease

In [3]:
LEASE_NAME = f"forkwise-k8s-{PROJECT_PREFIX}"

l = lease.Lease(
    LEASE_NAME,
    end_date=LEASE_END,
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id(VM_FLAVOR),
    amount=VM_COUNT,
)
l.submit(idempotent=True)

reservation_id = l.get_reserved_flavors()[0].id
print(f"Lease: {LEASE_NAME}")
print(f"Reservation flavor ID: {reservation_id}")

Waiting for lease to start...


Lease forkwise-k8s-proj01 has reached status active
Lease: forkwise-k8s-proj01
Reservation flavor ID: 3f9f99ed-dad3-492f-a7c3-9ffb0a02c5e8


## Step 4: Set Up Terraform

In [4]:
INFRA_REPO = "https://github.com/HivanshD/ml-sys-ops-project.git"
infra_dir = "/work/ml-sys-ops-project"
tf_dir = f"{infra_dir}/infra/tf/kvm"

# Clone or pull
if os.path.exists(infra_dir):
    result = subprocess.run("git pull", shell=True, cwd=infra_dir, capture_output=True, text=True)
    print(f"Repo updated: {result.stdout.strip()}")
else:
    result = subprocess.run(f"git clone {INFRA_REPO} {infra_dir}", shell=True, capture_output=True, text=True)
    print(f"Repo cloned")

# Verify clouds.yaml
if os.path.exists(f"{tf_dir}/clouds.yaml"):
    print(f"clouds.yaml found in {tf_dir}")
else:
    print(f"ERROR: clouds.yaml not found in {tf_dir}")

# Verify Terraform files
tf_files = [f for f in os.listdir(tf_dir) if f.endswith('.tf')]
print(f"Terraform files: {', '.join(sorted(tf_files))}")

Repo updated: Already up to date.
clouds.yaml found in /work/ml-sys-ops-project/infra/tf/kvm
Terraform files: data.tf, main.tf, outputs.tf, provider.tf, variables.tf, versions.tf


## Step 5: Create Security Groups & Terraform Apply

In [6]:
# Build clean environment
clean_env = {k: v for k, v in os.environ.items() if not k.startswith("OS_")}
clean_env["OS_CLOUD"]              = "openstack"
clean_env["PATH"]                  = "/work/.local/bin:" + clean_env.get("PATH", "")
clean_env["HOME"]                  = os.environ.get("HOME", "/home/jovyan")
clean_env["TF_VAR_suffix"]         = PROJECT_PREFIX
clean_env["TF_VAR_key"]            = SSH_KEY_NAME
clean_env["TF_VAR_reservation"]    = reservation_id


def run_tf(command, description):
    print(f"\n{'='*60}")
    print(f"  {description}")
    print(f"{'='*60}")
    result = subprocess.run(
        command, shell=True, cwd=tf_dir, env=clean_env,
        capture_output=True, text=True
    )
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.stderr:
        print(result.stderr[-1000:] if len(result.stderr) > 1000 else result.stderr)
    if result.returncode != 0:
        raise Exception(f"{description} failed (rc={result.returncode})")
    return result


def _os(cmd):
    r = subprocess.run(cmd, shell=True, env=clean_env,
                       capture_output=True, text=True)
    return r.returncode, r.stdout.strip(), r.stderr.strip()


# Ensure required security groups exist
REQUIRED_SGS = {
    "default": None,
}

rc, existing, _ = _os("openstack security group list -f value -c Name")
existing = set(existing.splitlines())
for name in REQUIRED_SGS:
    if name in existing:
        print(f"  [ok]  {name}")
    else:
        print(f"  [missing]  {name} — this should already exist")

print()
run_tf("terraform init", "Terraform Init")
run_tf("terraform validate", "Terraform Validate")
run_tf("terraform plan", "Terraform Plan")

  [missing]  default — this should already exist


  Terraform Init
Initializing the backend...
Initializing provider plugins...
- Finding terraform-provider-openstack/openstack versions matching "~> 1.51.1"...
- Installing terraform-provider-openstack/openstack v1.51.1...
- Installed terraform-provider-openstack/openstack v1.51.1 (self-signed, key ID 4F80527A391BEFD2)
Partner and community providers are signed by their developers.
If you'd like to know more about provider signing, you can read about it here:
https://www.terraform.io/docs/cli/plugins/signing.html
Terraform has created a lock file .terraform.lock.hcl to record the provider
selections it made above. Include this file in your version control repository
so that Terraform can guarantee to make the same selections by default when
you run "terraform init" in the future.

Terraform has been successfully initialized!

You may now begin working with Terraform. Try running "terraform plan" to see
any changes that are required for

CompletedProcess(args='terraform plan', returncode=0, stdout='\x1b\x1bopenstack_networking_router_interface_v2.router_iface: Refreshing state... [id=0a25be85-6cc3-4dd2-b3b5-fb0f97124d52]\x1b\n\x1b\x1bopenstack_networking_network_v2.project_net: Refreshing state... [id=ede2e924-b39e-485b-90aa-2fd58e739895]\x1b\n\x1b\x1bdata.openstack_networking_network_v2.sharednet1: Reading...\x1b\x1b\n\x1b\x1bdata.openstack_networking_secgroup_v2.default: Reading...\x1b\x1b\n\x1b\x1bopenstack_networking_subnet_v2.project_subnet: Refreshing state... [id=ad45f2dc-5b28-4a31-a8d5-283c063675e3]\x1b\n\x1b\x1bopenstack_networking_router_v2.router: Refreshing state... [id=7664c19b-a8bd-44a6-8150-030aecf1043a]\x1b\n\x1b\x1bopenstack_networking_port_v2.project_ports["node2"]: Refreshing state... [id=63414129-3c7a-4980-b354-3b734f7aaedd]\x1b\n\x1b\x1bopenstack_networking_port_v2.project_ports["node1"]: Refreshing state... [id=24414f45-471a-423d-b15d-9f63199c003e]\x1b\n\x1b\x1bopenstack_networking_port_v2.project

## Step 6: Terraform Apply

Creates 3 VMs + network + floating IP. Takes 2-5 minutes.

In [17]:
# Clean up stale ports if any
_os("openstack port list --status DOWN -f value -c ID | xargs -r -n1 openstack port delete")

run_tf("terraform apply -auto-approve", "Terraform Apply")

# Extract outputs
result = subprocess.run(
    "terraform output -json",
    shell=True, cwd=tf_dir, env=clean_env,
    capture_output=True, text=True
)
outputs = json.loads(result.stdout)
floating_ip = outputs["floating_ip"]["value"]

print(f"\nFloating IP: {floating_ip}")


  Terraform Apply
de3"]: Creation complete after 12s [id=2797ff98-e177-434a-a007-faaa974c2b25]
openstack_compute_instance_v2.nodes["node1"]: Creation complete after 12s [id=e54fe790-37fd-4b48-b170-cbecd0f21118]
╷
│ Warning: Value for undeclared variable
│ 
│ The root module does not declare a variable named "flavor_id" but a value
│ was found in file "terraform.tfvars". If you meant to use this value, add a
│ "variable" block to the configuration.
│ 
│ To silence these warnings, use TF_VAR_... environment variables to provide
│ certain "global" settings to all configurations in your organization. To
│ reduce the verbosity of these warnings, use the -compact-warnings option.
╵
╷
│ Warning: Value for undeclared variable
│ 
│ The root module does not declare a variable named "sharednet_name" but a
│ value was found in file "terraform.tfvars". If you meant to use this value,
│ add a "variable" block to the configuration.
│ 
│ To silence these warnings, use TF_VAR_... environment variables

## Step 7: Next Steps — Commands for the Nodes

In [22]:
from IPython.display import display, Markdown

instructions = fr"""
---

# Part 2: Setup Cluster & Deploy

---

## A. Upload SSH key & connect

From **Windows PowerShell**:

```powershell
scp -i $HOME\.ssh\forkwise_key $HOME\.ssh\forkwise_key cc@{floating_ip}:~/.ssh/id_rsa
ssh -i $HOME\.ssh\forkwise_key cc@{floating_ip}
```

On node1:

```bash
chmod 600 ~/.ssh/id_rsa
```

---

## B. Pre-K8s Setup (run on node1 — automates all 3 nodes)

```bash
PREK8S='sudo apt-get update && sudo apt-get install -y curl ca-certificates jq python3 docker.io && \
sudo systemctl enable docker && sudo systemctl start docker && \
sudo usermod -aG docker $USER && \
(sudo systemctl stop ufw 2>/dev/null; sudo systemctl disable ufw 2>/dev/null; true) && \
echo "br_netfilter" | sudo tee /etc/modules-load.d/k8s.conf && \
sudo modprobe br_netfilter && \
echo -e "net.bridge.bridge-nf-call-iptables = 1\nnet.bridge.bridge-nf-call-ip6tables = 1\nnet.ipv4.ip_forward = 1" | sudo tee /etc/sysctl.d/99-kubernetes-cri.conf && \
sudo sysctl --system'

echo "=== Configuring node1 ==="
eval $PREK8S

for ip in 192.168.1.12 192.168.1.13; do
  echo "=== Configuring $ip ==="
  ssh -o StrictHostKeyChecking=no cc@$ip "$PREK8S"
done

echo "=== All nodes configured ==="
```

---

## C. Install k3s (run on node1 — automates all 3 nodes)

```bash
# Install server on node1
curl -sfL https://get.k3s.io | \
  INSTALL_K3S_EXEC="server --write-kubeconfig-mode 644 --node-ip 192.168.1.11 --advertise-address 192.168.1.11 --tls-san 192.168.1.11" \
  sh -

# Get token
K3S_TOKEN=$(sudo cat /var/lib/rancher/k3s/server/node-token)

# Join workers
for ip in 192.168.1.12 192.168.1.13; do
  echo "=== Joining $ip ==="
  ssh -o StrictHostKeyChecking=no cc@$ip \
    "curl -sfL https://get.k3s.io | K3S_URL=https://192.168.1.11:6443 K3S_TOKEN=$K3S_TOKEN INSTALL_K3S_EXEC='agent --node-ip $ip' sh -"
done

# Verify
sudo kubectl get nodes
```

Wait until all 3 nodes show `Ready` (1-2 min):

```bash
until [ $(sudo kubectl get nodes --no-headers | grep -c Ready) -eq 3 ]; do
  echo "Waiting for all nodes to be Ready..."
  sleep 10
done
echo "All 3 nodes Ready"
sudo kubectl get nodes
```

---

## D. Post-k3s Setup (node1)

```bash
mkdir -p ~/.kube
sudo cp /etc/rancher/k3s/k3s.yaml ~/.kube/config
sudo chown $(id -u):$(id -g) ~/.kube/config
sed -i 's|https://127.0.0.1:6443|https://192.168.1.11:6443|' ~/.kube/config

curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash

kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
kubectl patch deployment metrics-server -n kube-system --type=json \
  -p='[{{"op":"add","path":"/spec/template/spec/containers/0/args/-","value":"--kubelet-insecure-tls"}}]'

kubectl label node $(hostname) forkwise.io/entrypoint=true --overwrite
kubectl get nodes
```

---

## E. Clone Repos & Copy Manifests (node1)

```bash
cd ~
git clone https://github.com/HivanshD/ml-sys-ops-project.git
git clone https://github.com/HivanshD/mealie.git
sudo cp -r ~/ml-sys-ops-project/infra/k8s /tmp/serving-k8s
```

---

## F. Build Docker Images (node1)

```bash
cd ~/ml-sys-ops-project/data
sudo docker build -t forkwise-ingest:local -f Dockerfile.ingest .
sudo docker build -t forkwise-feedback:local -f Dockerfile.feedback .
sudo docker build -t forkwise-batch:local -f Dockerfile.batch .
sudo docker build -t forkwise-generator:local -f Dockerfile.generator .

cd ~/ml-sys-ops-project/serving
sudo docker build -t subst-serving-onnx:local -f docker/Dockerfile.fastapi_onnx .

cd ~/ml-sys-ops-project
sudo docker build -t forkwise-train:local -f training/docker_nvidia/Dockerfile .

cd ~/ml-sys-ops-project/infra/automation
sudo docker build -t forkwise-automation:local .

cd ~/mealie
sudo docker build -t mealie:ml-ui -f docker/Dockerfile .
```

Import into k3s:

```bash
for img in forkwise-ingest:local forkwise-feedback:local forkwise-batch:local \
           forkwise-generator:local subst-serving-onnx:local forkwise-train:local \
           forkwise-automation:local mealie:ml-ui; do
  echo "Importing $img..."
  sudo docker save $img | sudo k3s ctr images import -
done
```

---

## G. Patch Manifests for Local Images (node1)

```bash
cd /tmp/serving-k8s

sed -i 's|ghcr.io/itsnotaka/forkwise-ingest:demo|forkwise-ingest:local|g' $(grep -rl 'forkwise-ingest' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/forkwise-feedback:demo|forkwise-feedback:local|g' $(grep -rl 'forkwise-feedback' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/forkwise-batch:demo|forkwise-batch:local|g' $(grep -rl 'forkwise-batch' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/forkwise-generator:demo|forkwise-generator:local|g' $(grep -rl 'forkwise-generator' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/mealie:ml-ui-amd64|mealie:ml-ui|g' $(grep -rl 'mealie:ml-ui' . 2>/dev/null) 2>/dev/null

find . -name '*.yaml' | xargs grep -l ':local' 2>/dev/null | while read f; do
  sed -i '/:local/a\        imagePullPolicy: Never' "$f"
done
find . -name '*.yaml' | xargs grep -l 'mealie:ml-ui' 2>/dev/null | while read f; do
  sed -i '/mealie:ml-ui/a\        imagePullPolicy: Never' "$f"
done
```

---

## H. Deploy Apps (node1)

```bash
kubectl apply -f /tmp/serving-k8s/apps/mealie/namespace.yaml
kubectl apply -f /tmp/serving-k8s/apps/substitution-serving/namespace.yaml

kubectl create secret generic mealie-credentials \
  --namespace forkwise-app \
  --from-literal=postgres-user=mealie \
  --from-literal=postgres-password=$(openssl rand -base64 20) \
  --from-literal=postgres-db=mealie \
  --from-literal=base-url=http://localhost:9000

kubectl apply -k /tmp/serving-k8s/apps/substitution-serving
kubectl apply -k /tmp/serving-k8s/apps/mealie

kubectl set image deployment/substitution-serving \
  substitution-serving=subst-serving-onnx:local -n forkwise-serving
kubectl set image cronjob/check-rollback \
  check-rollback=subst-serving-onnx:local -n forkwise-serving

kubectl rollout status deployment/substitution-serving -n forkwise-serving --timeout=180s
kubectl rollout status deployment/mealie-postgres -n forkwise-app --timeout=180s
kubectl rollout status deployment/mealie -n forkwise-app --timeout=240s
```

---

## I. Create Secrets (node1)

```bash
OS_AK="{OS_ACCESS_KEY}"
OS_SK="{OS_SECRET_KEY}"

for ns in monitoring-proj01 staging-proj01 canary-proj01 production-proj01; do
  kubectl create namespace $ns --dry-run=client -o yaml | kubectl apply -f -
  kubectl create secret generic os-credentials -n $ns \
    --from-literal=OS_ENDPOINT=https://chi.tacc.chameleoncloud.org:7480 \
    --from-literal=OS_ACCESS_KEY=$OS_AK \
    --from-literal=OS_SECRET_KEY=$OS_SK \
    --dry-run=client -o yaml | kubectl apply -f -
done

for ns in forkwise-data forkwise-serving; do
  kubectl create namespace $ns --dry-run=client -o yaml | kubectl apply -f -
  kubectl create secret generic s3-credentials -n $ns \
    --from-literal=access-key=$OS_AK \
    --from-literal=secret-key=$OS_SK \
    --dry-run=client -o yaml | kubectl apply -f -
done
```

---

## J. Deploy Rollout Stack (node1)

```bash
kubectl apply -k /tmp/serving-k8s/platform

kubectl set image deployment/automation \
  automation=forkwise-automation:local -n monitoring-proj01

kubectl rollout status deployment/prometheus -n monitoring-proj01 --timeout=180s
kubectl rollout status deployment/grafana -n monitoring-proj01 --timeout=180s
kubectl rollout status deployment/automation -n monitoring-proj01 --timeout=180s

kubectl exec -n monitoring-proj01 deploy/automation -- \
  curl -fsS -X POST http://127.0.0.1:8080/bootstrap-rollout

for env in staging canary production; do
  kubectl apply -k /tmp/serving-k8s/$env
  kubectl set image deployment/subst-serving \
    subst-serving=subst-serving-onnx:local -n ${{env}}-proj01
  kubectl rollout restart deployment/subst-serving -n ${{env}}-proj01
done
```

---

## K. Deploy Data Workloads (node1)

```bash
kubectl apply -k /tmp/serving-k8s/apps/forkwise-data

kubectl apply -f /tmp/serving-k8s/apps/forkwise-data/job-ingest.yaml
kubectl logs -n forkwise-data job/forkwise-ingest -f
kubectl wait --for=condition=complete job/forkwise-ingest -n forkwise-data --timeout=1800s

kubectl scale deployment/data-generator -n forkwise-data --replicas=1
kubectl patch cronjob batch-pipeline -n forkwise-data -p '{{"spec":{{"suspend":false}}}}'
kubectl patch cronjob drift-monitor -n forkwise-data -p '{{"spec":{{"suspend":false}}}}'
```

---

## L. Smoke Test (node1)

```bash
kubectl get nodes
kubectl get pods -A
curl -sf http://127.0.0.1:30080/health && echo "Serving OK" || echo "Serving not ready"
curl -sf http://127.0.0.1:30090 -o /dev/null && echo "Mealie OK" || echo "Mealie not ready"
```

---

## M. Access from Windows

```powershell
ssh -i $HOME\.ssh\forkwise_key -N -L 9000:127.0.0.1:30090 cc@{floating_ip}
ssh -i $HOME\.ssh\forkwise_key -N -L 8000:127.0.0.1:30080 cc@{floating_ip}
ssh -i $HOME\.ssh\forkwise_key -N -L 3000:127.0.0.1:30300 cc@{floating_ip}
ssh -i $HOME\.ssh\forkwise_key -N -L 9090:127.0.0.1:30900 cc@{floating_ip}
```

- **Mealie:** http://localhost:9000
- **Serving:** http://localhost:8000/health
- **Grafana:** http://localhost:3000
- **Prometheus:** http://localhost:9090
"""

display(Markdown(instructions))


---

# Part 2: Setup Cluster & Deploy

---

## A. Upload SSH key & connect

From **Windows PowerShell**:

```powershell
scp -i $HOME\.ssh\forkwise_key $HOME\.ssh\forkwise_key cc@129.114.27.91:~/.ssh/id_rsa
ssh -i $HOME\.ssh\forkwise_key cc@129.114.27.91
```

On node1:

```bash
chmod 600 ~/.ssh/id_rsa
```

---

## B. Pre-K8s Setup (run on node1 — automates all 3 nodes)

```bash
PREK8S='sudo apt-get update && sudo apt-get install -y curl ca-certificates jq python3 docker.io && \
sudo systemctl enable docker && sudo systemctl start docker && \
sudo usermod -aG docker $USER && \
(sudo systemctl stop ufw 2>/dev/null; sudo systemctl disable ufw 2>/dev/null; true) && \
echo "br_netfilter" | sudo tee /etc/modules-load.d/k8s.conf && \
sudo modprobe br_netfilter && \
echo -e "net.bridge.bridge-nf-call-iptables = 1\nnet.bridge.bridge-nf-call-ip6tables = 1\nnet.ipv4.ip_forward = 1" | sudo tee /etc/sysctl.d/99-kubernetes-cri.conf && \
sudo sysctl --system'

echo "=== Configuring node1 ==="
eval $PREK8S

for ip in 192.168.1.12 192.168.1.13; do
  echo "=== Configuring $ip ==="
  ssh -o StrictHostKeyChecking=no cc@$ip "$PREK8S"
done

echo "=== All nodes configured ==="
```

---

## C. Install k3s (run on node1 — automates all 3 nodes)

```bash
# Install server on node1
curl -sfL https://get.k3s.io | \
  INSTALL_K3S_EXEC="server --write-kubeconfig-mode 644 --node-ip 192.168.1.11 --advertise-address 192.168.1.11 --tls-san 192.168.1.11" \
  sh -

# Get token
K3S_TOKEN=$(sudo cat /var/lib/rancher/k3s/server/node-token)

# Join workers
for ip in 192.168.1.12 192.168.1.13; do
  echo "=== Joining $ip ==="
  ssh -o StrictHostKeyChecking=no cc@$ip \
    "curl -sfL https://get.k3s.io | K3S_URL=https://192.168.1.11:6443 K3S_TOKEN=$K3S_TOKEN INSTALL_K3S_EXEC='agent --node-ip $ip' sh -"
done

# Verify
sudo kubectl get nodes
```

Wait until all 3 nodes show `Ready` (1-2 min):

```bash
until [ $(sudo kubectl get nodes --no-headers | grep -c Ready) -eq 3 ]; do
  echo "Waiting for all nodes to be Ready..."
  sleep 10
done
echo "All 3 nodes Ready"
sudo kubectl get nodes
```

---

## D. Post-k3s Setup (node1)

```bash
mkdir -p ~/.kube
sudo cp /etc/rancher/k3s/k3s.yaml ~/.kube/config
sudo chown $(id -u):$(id -g) ~/.kube/config
sed -i 's|https://127.0.0.1:6443|https://192.168.1.11:6443|' ~/.kube/config

curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash

kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml
kubectl patch deployment metrics-server -n kube-system --type=json \
  -p='[{"op":"add","path":"/spec/template/spec/containers/0/args/-","value":"--kubelet-insecure-tls"}]'

kubectl label node $(hostname) forkwise.io/entrypoint=true --overwrite
kubectl get nodes
```

---

## E. Clone Repos & Copy Manifests (node1)

```bash
cd ~
git clone https://github.com/HivanshD/ml-sys-ops-project.git
git clone https://github.com/HivanshD/mealie.git
sudo cp -r ~/ml-sys-ops-project/infra/k8s /tmp/serving-k8s
```

---

## F. Build Docker Images (node1)

```bash
cd ~/ml-sys-ops-project/data
sudo docker build -t forkwise-ingest:local -f Dockerfile.ingest .
sudo docker build -t forkwise-feedback:local -f Dockerfile.feedback .
sudo docker build -t forkwise-batch:local -f Dockerfile.batch .
sudo docker build -t forkwise-generator:local -f Dockerfile.generator .

cd ~/ml-sys-ops-project/serving
sudo docker build -t subst-serving-onnx:local -f docker/Dockerfile.fastapi_onnx .

cd ~/ml-sys-ops-project
sudo docker build -t forkwise-train:local -f training/docker_nvidia/Dockerfile .

cd ~/ml-sys-ops-project/infra/automation
sudo docker build -t forkwise-automation:local .

cd ~/mealie
sudo docker build -t mealie:ml-ui -f docker/Dockerfile .
```

Import into k3s:

```bash
for img in forkwise-ingest:local forkwise-feedback:local forkwise-batch:local \
           forkwise-generator:local subst-serving-onnx:local forkwise-train:local \
           forkwise-automation:local mealie:ml-ui; do
  echo "Importing $img..."
  sudo docker save $img | sudo k3s ctr images import -
done
```

---

## G. Patch Manifests for Local Images (node1)

```bash
cd /tmp/serving-k8s

sed -i 's|ghcr.io/itsnotaka/forkwise-ingest:demo|forkwise-ingest:local|g' $(grep -rl 'forkwise-ingest' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/forkwise-feedback:demo|forkwise-feedback:local|g' $(grep -rl 'forkwise-feedback' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/forkwise-batch:demo|forkwise-batch:local|g' $(grep -rl 'forkwise-batch' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/forkwise-generator:demo|forkwise-generator:local|g' $(grep -rl 'forkwise-generator' . 2>/dev/null) 2>/dev/null
sed -i 's|ghcr.io/itsnotaka/mealie:ml-ui-amd64|mealie:ml-ui|g' $(grep -rl 'mealie:ml-ui' . 2>/dev/null) 2>/dev/null

find . -name '*.yaml' | xargs grep -l ':local' 2>/dev/null | while read f; do
  sed -i '/:local/a\        imagePullPolicy: Never' "$f"
done
find . -name '*.yaml' | xargs grep -l 'mealie:ml-ui' 2>/dev/null | while read f; do
  sed -i '/mealie:ml-ui/a\        imagePullPolicy: Never' "$f"
done
```

---

## H. Deploy Apps (node1)

```bash
kubectl apply -f /tmp/serving-k8s/apps/mealie/namespace.yaml
kubectl apply -f /tmp/serving-k8s/apps/substitution-serving/namespace.yaml

kubectl create secret generic mealie-credentials \
  --namespace forkwise-app \
  --from-literal=postgres-user=mealie \
  --from-literal=postgres-password=$(openssl rand -base64 20) \
  --from-literal=postgres-db=mealie \
  --from-literal=base-url=http://localhost:9000

kubectl apply -k /tmp/serving-k8s/apps/substitution-serving
kubectl apply -k /tmp/serving-k8s/apps/mealie

kubectl set image deployment/substitution-serving \
  substitution-serving=subst-serving-onnx:local -n forkwise-serving
kubectl set image cronjob/check-rollback \
  check-rollback=subst-serving-onnx:local -n forkwise-serving

kubectl rollout status deployment/substitution-serving -n forkwise-serving --timeout=180s
kubectl rollout status deployment/mealie-postgres -n forkwise-app --timeout=180s
kubectl rollout status deployment/mealie -n forkwise-app --timeout=240s
```

---

## I. Create Secrets (node1)

```bash
OS_AK="8921c48faf83433db2b1439a9b2889fd"
OS_SK="7d1ce78efc5a48019888c9f3fa8ba2dd"

for ns in monitoring-proj01 staging-proj01 canary-proj01 production-proj01; do
  kubectl create namespace $ns --dry-run=client -o yaml | kubectl apply -f -
  kubectl create secret generic os-credentials -n $ns \
    --from-literal=OS_ENDPOINT=https://chi.tacc.chameleoncloud.org:7480 \
    --from-literal=OS_ACCESS_KEY=$OS_AK \
    --from-literal=OS_SECRET_KEY=$OS_SK \
    --dry-run=client -o yaml | kubectl apply -f -
done

for ns in forkwise-data forkwise-serving; do
  kubectl create namespace $ns --dry-run=client -o yaml | kubectl apply -f -
  kubectl create secret generic s3-credentials -n $ns \
    --from-literal=access-key=$OS_AK \
    --from-literal=secret-key=$OS_SK \
    --dry-run=client -o yaml | kubectl apply -f -
done
```

---

## J. Deploy Rollout Stack (node1)

```bash
kubectl apply -k /tmp/serving-k8s/platform

kubectl set image deployment/automation \
  automation=forkwise-automation:local -n monitoring-proj01

kubectl rollout status deployment/prometheus -n monitoring-proj01 --timeout=180s
kubectl rollout status deployment/grafana -n monitoring-proj01 --timeout=180s
kubectl rollout status deployment/automation -n monitoring-proj01 --timeout=180s

kubectl exec -n monitoring-proj01 deploy/automation -- \
  curl -fsS -X POST http://127.0.0.1:8080/bootstrap-rollout

for env in staging canary production; do
  kubectl apply -k /tmp/serving-k8s/$env
  kubectl set image deployment/subst-serving \
    subst-serving=subst-serving-onnx:local -n ${env}-proj01
  kubectl rollout restart deployment/subst-serving -n ${env}-proj01
done
```

---

## K. Deploy Data Workloads (node1)

```bash
kubectl apply -k /tmp/serving-k8s/apps/forkwise-data

kubectl apply -f /tmp/serving-k8s/apps/forkwise-data/job-ingest.yaml
kubectl logs -n forkwise-data job/forkwise-ingest -f
kubectl wait --for=condition=complete job/forkwise-ingest -n forkwise-data --timeout=1800s

kubectl scale deployment/data-generator -n forkwise-data --replicas=1
kubectl patch cronjob batch-pipeline -n forkwise-data -p '{"spec":{"suspend":false}}'
kubectl patch cronjob drift-monitor -n forkwise-data -p '{"spec":{"suspend":false}}'
```

---

## L. Smoke Test (node1)

```bash
kubectl get nodes
kubectl get pods -A
curl -sf http://127.0.0.1:30080/health && echo "Serving OK" || echo "Serving not ready"
curl -sf http://127.0.0.1:30090 -o /dev/null && echo "Mealie OK" || echo "Mealie not ready"
```

---

## M. Access from Windows

```powershell
ssh -i $HOME\.ssh\forkwise_key -N -L 9000:127.0.0.1:30090 cc@129.114.27.91
ssh -i $HOME\.ssh\forkwise_key -N -L 8000:127.0.0.1:30080 cc@129.114.27.91
ssh -i $HOME\.ssh\forkwise_key -N -L 3000:127.0.0.1:30300 cc@129.114.27.91
ssh -i $HOME\.ssh\forkwise_key -N -L 9090:127.0.0.1:30900 cc@129.114.27.91
```

- **Mealie:** http://localhost:9000
- **Serving:** http://localhost:8000/health
- **Grafana:** http://localhost:3000
- **Prometheus:** http://localhost:9090


## Teardown

Uncomment and run to destroy everything.

In [ ]:
# run_tf("terraform destroy -auto-approve", "Terraform Destroy")
# l.delete()
# print("Lease deleted, VMs destroyed.")